# DeepSets Extrapolation Training

This notebook trains DeepSets models on mixed-distance datasets (d=3, d=5, d=7) to study extrapolation to larger distances (d=9, d=11, d=13).

**Purpose:** Benchmark DeepSets (variable-size NN) against GraphSAGE for extrapolation.

**Key Advantage:** DeepSets handles variable-size inputs natively through permutation-invariant
aggregation - no padding or truncation needed for different distances!

**Workflow:**
1. Load best hyperparameters from tuning results
2. Configure split experiments (different ratios of d=3/d=5/d=7 data)
3. Generate/load combined training datasets (flat syndrome arrays)
4. Train models for each split configuration
5. Save trained models for evaluation in testing.ipynb

**Split Experiments:**

| Split Name | d=3 | d=5 | d=7 | Hypothesis |
|------------|-----|-----|-----|------------|
| baseline_33_33_33 | 33% | 33% | 34% | Reference baseline |
| no_d3_50_50 | 0% | 50% | 50% | Is d=3 useless? |
| no_d3_30_70 | 0% | 30% | 70% | Does more d=7 help? |
| tiny_d3_10_40_50 | 10% | 40% | 50% | Does tiny d=3 help? |
| reversed_50_30_20 | 50% | 30% | 20% | Sanity check: direction matters? |

## Imports

In [ ]:
# Install required libraries (uncomment and run if needed)
# !pip install stim pymatching numpy matplotlib torch tqdm

In [ ]:
import sys
import json
import random
import time
from pathlib import Path
from datetime import datetime

# Detect if running in Google Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = Path('/content/drive/MyDrive/Research/QEC/quantum-error-correction/quantum-error-correction/code')
else:
    BASE_PATH = Path('../..')  # code/deepsets/extrapolation -> code/

sys.path.insert(0, str(BASE_PATH))

import torch
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from tqdm import tqdm

# Import from benchmark_models.py
from benchmark_models import (
    SurfaceCodeSampler,
    DeepSetsModel,
    DeepSets,
    FlatDatasetCache,
)

# Set up paths
EXTRAPOLATION_DIR = BASE_PATH / "deepsets" / "extrapolation"
EXTRAPOLATION_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR = EXTRAPOLATION_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR = EXTRAPOLATION_DIR / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR = EXTRAPOLATION_DIR / "plots"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

print(f"\nPaths:")
print(f"  BASE_PATH: {BASE_PATH}")
print(f"  EXTRAPOLATION_DIR: {EXTRAPOLATION_DIR}")
print(f"  RESULTS_DIR: {RESULTS_DIR}")
print(f"  MODELS_DIR: {MODELS_DIR}")

## Configuration

In [ ]:
# =============================================================================
# BEST HYPERPARAMETERS (loaded from tuning results)
# =============================================================================

# Load best hyperparameters from tuning results
BEST_CONFIG_PATH = BASE_PATH / "deepsets" / "tuning" / "best_model_config.json"

if BEST_CONFIG_PATH.exists():
    with open(BEST_CONFIG_PATH, 'r') as f:
        best_config = json.load(f)

    BEST_HYPERPARAMS = {
        'phi_hidden': tuple(best_config['phi_hidden']) if isinstance(best_config['phi_hidden'], list) else best_config['phi_hidden'],
        'rho_hidden': tuple(best_config['rho_hidden']) if isinstance(best_config['rho_hidden'], list) else best_config['rho_hidden'],
        'pool': best_config['pool'],
        'learning_rate': best_config['learning_rate'],
        'batch_size': best_config['batch_size'],
        'dropout': best_config['dropout'],
    }

    print(f"Loaded best hyperparameters from: {BEST_CONFIG_PATH}")
    print(f"Config ID: {best_config.get('config_id', 'N/A')}")
    print(f"Tuning performance: test_acc={best_config.get('test_accuracy', 0)*100:.2f}%")
else:
    # Default hyperparameters if tuning hasn't been run yet
    print(f"WARNING: Best config not found at {BEST_CONFIG_PATH}")
    print("Using default hyperparameters. Run tuning.ipynb first for optimal results.")
    BEST_HYPERPARAMS = {
        'phi_hidden': (128, 128),
        'rho_hidden': (256, 128),
        'pool': 'mean',
        'learning_rate': 0.0003,
        'batch_size': 64,
        'dropout': 0.1,
    }

print(f"\nHyperparameters:")
for k, v in BEST_HYPERPARAMS.items():
    print(f"  {k}: {v}")

In [ ]:
# =============================================================================
# TRAINING CONFIGURATION
# =============================================================================

# Total training samples (matching gSAGE: 10^6)
TOTAL_SAMPLES = 1_000_000

# Training parameters
EPOCHS = 50

# Error rate distribution for training data
P_VALUES = [0.001, 0.003, 0.005, 0.007]
P_WEIGHTS = [0.25, 0.25, 0.25, 0.25]

# Random seed for reproducibility
SEED = 42

print(f"Training Configuration:")
print(f"  Total samples: {TOTAL_SAMPLES:,}")
print(f"  Epochs: {EPOCHS}")
print(f"  Batch size: {BEST_HYPERPARAMS['batch_size']}")
print(f"  P values: {P_VALUES}")
print(f"  P weights: {P_WEIGHTS}")

In [ ]:
# =============================================================================
# SPLIT EXPERIMENT CONFIGURATIONS
# =============================================================================

SPLIT_EXPERIMENTS = {
    # Reference baseline - equal distribution
    'baseline_33_33_33': {
        'd3': 0.33,
        'd5': 0.33,
        'd7': 0.34,
        'hypothesis': 'Reference baseline for comparison'
    },

    # Is d=3 useless? Test if small codes just add noise
    'no_d3_50_50': {
        'd3': 0.00,
        'd5': 0.50,
        'd7': 0.50,
        'hypothesis': 'Is d=3 useless? Tests if small codes add noise'
    },

    # Does more d=7 help extrapolation?
    'no_d3_30_70': {
        'd3': 0.00,
        'd5': 0.30,
        'd7': 0.70,
        'hypothesis': 'Does more d=7 help extrapolation to larger distances?'
    },

    # Does even a tiny amount of d=3 help?
    'tiny_d3_10_40_50': {
        'd3': 0.10,
        'd5': 0.40,
        'd7': 0.50,
        'hypothesis': 'Does tiny d=3 help? Tests minimal contribution'
    },

    # Sanity check - direction should matter
    'reversed_50_30_20': {
        'd3': 0.50,
        'd5': 0.30,
        'd7': 0.20,
        'hypothesis': 'Sanity check: direction should matter (expected worst)'
    },
}

print("Split Experiments:")
print(f"{'Name':<25} {'d3':>6} {'d5':>6} {'d7':>6}   Hypothesis")
print("-" * 80)
for name, config in SPLIT_EXPERIMENTS.items():
    print(f"{name:<25} {config['d3']*100:>5.0f}% {config['d5']*100:>5.0f}% {config['d7']*100:>5.0f}%   {config['hypothesis']}")

In [ ]:
# =============================================================================
# SELECT WHICH EXPERIMENT TO RUN
# =============================================================================

# Set to None to run all experiments, or specify a list of experiment names
# Examples:
#   EXPERIMENTS_TO_RUN = None  # Run all
#   EXPERIMENTS_TO_RUN = ['baseline_33_33_33']  # Run only baseline
#   EXPERIMENTS_TO_RUN = ['baseline_33_33_33', 'no_d3_50_50']  # Run two experiments

EXPERIMENTS_TO_RUN = None  # Run all experiments

if EXPERIMENTS_TO_RUN is None:
    experiments_to_run = list(SPLIT_EXPERIMENTS.keys())
else:
    experiments_to_run = EXPERIMENTS_TO_RUN

print(f"Experiments to run: {experiments_to_run}")

## Helper Functions

In [ ]:
def generate_flat_dataset_for_distance(d: int, n_samples: int, cache_name: str = None):
    """
    Generate or load a flat dataset for a specific distance.

    Args:
        d: Code distance
        n_samples: Number of samples to generate
        cache_name: Optional cache name to load from/save to

    Returns:
        Tuple of (detections, labels) tensors
    """
    cache = FlatDatasetCache(base_path=BASE_PATH, device=device)

    # Try to load from cache first
    if cache_name:
        try:
            cache.load(cache_name, verbose=True)
            # Ensure we have enough samples
            if len(cache) >= n_samples:
                print(f"Using {n_samples:,} samples from cache '{cache_name}'")
                return cache.get_data(n=n_samples, shuffle=True)
            else:
                print(f"Cache has {len(cache):,} samples, need {n_samples:,}. Regenerating...")
        except FileNotFoundError:
            print(f"Cache '{cache_name}' not found. Generating new dataset...")

    # Generate new dataset
    cache.generate(
        d=d,
        n_samples=n_samples,
        p_values=P_VALUES,
        p_weights=P_WEIGHTS,
        verbose=True
    )

    # Save to cache if name provided
    if cache_name:
        cache.save(cache_name)

    return cache.get_data(shuffle=True)

In [ ]:
def load_mixed_flat_dataset_variable(split_config: dict, total_samples: int):
    """
    Load and combine flat datasets for DeepSets training.

    Unlike SimpleNN, DeepSets handles variable-size inputs, so we:
    1. Load each distance separately
    2. Pad to max size within the combined dataset
    3. Create masks to indicate valid detectors

    Args:
        split_config: Dict with 'd3', 'd5', 'd7' keys specifying proportions
        total_samples: Total number of samples in combined dataset

    Returns:
        Tuple of (padded_detections, labels, masks)
    """
    all_detections = []  # List of (detections_tensor, num_detectors)
    all_labels = []
    detector_counts = []

    for distance_key, distance in [('d3', 3), ('d5', 5), ('d7', 7)]:
        proportion = split_config[distance_key]
        if proportion > 0:
            n_samples = int(total_samples * proportion)
            if n_samples > 0:
                cache_name = f"d{distance}_baseline"
                print(f"\nLoading d={distance} dataset ({n_samples:,} samples, {proportion*100:.0f}%)...")
                detections, labels = generate_flat_dataset_for_distance(distance, n_samples, cache_name)

                num_detectors = detections.shape[1]
                all_detections.append((detections, num_detectors))
                all_labels.append(labels)
                detector_counts.extend([num_detectors] * len(labels))
                print(f"  Added {len(labels):,} d={distance} samples ({num_detectors} detectors each)")

    # Find max detector count for padding
    max_detectors = max(det[1] for det in all_detections)
    print(f"\nMax detectors: {max_detectors} (will pad smaller distances)")

    # Pad and combine
    padded_list = []
    mask_list = []

    for detections, num_detectors in all_detections:
        n = detections.shape[0]
        if num_detectors < max_detectors:
            # Pad with zeros
            padding = torch.zeros(n, max_detectors - num_detectors, device=device)
            padded = torch.cat([detections, padding], dim=1)
        else:
            padded = detections

        # Create mask (True for valid, False for padding)
        mask = torch.zeros(n, max_detectors, dtype=torch.bool, device=device)
        mask[:, :num_detectors] = True

        padded_list.append(padded)
        mask_list.append(mask)

    # Concatenate all
    combined_detections = torch.cat(padded_list, dim=0)
    combined_labels = torch.cat(all_labels, dim=0)
    combined_masks = torch.cat(mask_list, dim=0)

    # Shuffle combined dataset
    torch.manual_seed(SEED)
    perm = torch.randperm(len(combined_labels), device=device)
    combined_detections = combined_detections[perm]
    combined_labels = combined_labels[perm]
    combined_masks = combined_masks[perm]

    print(f"\nCombined dataset: {len(combined_labels):,} total samples")
    print(f"  Detections shape: {combined_detections.shape}")
    print(f"  Masks shape: {combined_masks.shape}")

    return combined_detections, combined_labels, combined_masks

In [ ]:
def split_train_val_test(detections, labels, masks, train_ratio: float = 0.8, val_ratio: float = 0.1, seed: int = 42):
    """
    Split dataset into train/val/test sets.

    Args:
        detections: Tensor of shape [n_samples, num_detectors]
        labels: Tensor of shape [n_samples]
        masks: Tensor of shape [n_samples, num_detectors]
        train_ratio: Proportion for training
        val_ratio: Proportion for validation
        seed: Random seed

    Returns:
        Tuple of train/val/test data tuples
    """
    torch.manual_seed(seed)
    n = len(labels)
    perm = torch.randperm(n, device=labels.device)

    train_end = int(n * train_ratio)
    val_end = int(n * (train_ratio + val_ratio))

    train_idx = perm[:train_end]
    val_idx = perm[train_end:val_end]
    test_idx = perm[val_end:]

    train_data = (detections[train_idx], labels[train_idx], masks[train_idx])
    val_data = (detections[val_idx], labels[val_idx], masks[val_idx])
    test_data = (detections[test_idx], labels[test_idx], masks[test_idx])

    print(f"Dataset split: {len(train_idx):,} train, {len(val_idx):,} val, {len(test_idx):,} test")

    return train_data, val_data, test_data

In [ ]:
def evaluate_deepsets(model, detections, labels, masks, batch_size=256):
    """
    Evaluate DeepSets model accuracy on a set of samples.

    Args:
        model: DeepSets model wrapper
        detections: Tensor of shape [n_samples, num_detectors]
        labels: Tensor of shape [n_samples]
        masks: Tensor of shape [n_samples, num_detectors]
        batch_size: Batch size for evaluation

    Returns:
        Accuracy as a float
    """
    model.model.eval()

    correct = 0
    total = len(labels)

    with torch.no_grad():
        for i in range(0, total, batch_size):
            X = detections[i:i+batch_size].unsqueeze(-1)  # [batch, N, 1]
            y = labels[i:i+batch_size]
            m = masks[i:i+batch_size]
            pred = model.model(X, m)
            correct += ((pred.squeeze() > 0.5).float() == y).sum().item()

    return correct / total if total > 0 else 0.0

In [ ]:
def save_model(model, split_name: str, metrics: dict):
    """
    Save trained model with metadata.

    Args:
        model: DeepSets model wrapper
        split_name: Name of the split experiment
        metrics: Dict with training metrics
    """
    model_path = MODELS_DIR / f"{split_name}.pt"

    save_dict = {
        'state_dict': model.model.state_dict(),
        'config': model._config,
        'split_name': split_name,
        'split_config': SPLIT_EXPERIMENTS[split_name],
        'hyperparams': BEST_HYPERPARAMS,
        'metrics': metrics,
        'timestamp': datetime.now().isoformat(),
    }

    torch.save(save_dict, model_path)
    print(f"Model saved to: {model_path}")
    return model_path

In [ ]:
def save_training_results(split_name: str, metrics: dict):
    """
    Save training results to JSON.

    Args:
        split_name: Name of the split experiment
        metrics: Dict with training metrics
    """
    results_path = RESULTS_DIR / f"{split_name}_training.json"

    results = {
        'split_name': split_name,
        'split_config': SPLIT_EXPERIMENTS[split_name],
        'hyperparams': BEST_HYPERPARAMS,
        'training_config': {
            'total_samples': TOTAL_SAMPLES,
            'epochs': EPOCHS,
            'batch_size': BEST_HYPERPARAMS['batch_size'],
            'p_values': P_VALUES,
            'p_weights': P_WEIGHTS,
        },
        'metrics': metrics,
        'timestamp': datetime.now().isoformat(),
    }

    with open(results_path, 'w') as f:
        json.dump(results, f, indent=2, default=str)

    print(f"Results saved to: {results_path}")

## Training Pipeline

In [ ]:
def train_deepsets(train_det, train_lab, train_mask, epochs, batch_size, lr, verbose=True):
    """
    Train a DeepSets model.

    Args:
        train_det: Training detections tensor
        train_lab: Training labels tensor
        train_mask: Training masks tensor
        epochs: Number of epochs
        batch_size: Batch size
        lr: Learning rate
        verbose: Print progress

    Returns:
        Tuple of (model, epoch_losses)
    """
    # Create model
    model = DeepSets(
        nickname="extrapolation",
        phi_hidden=BEST_HYPERPARAMS['phi_hidden'],
        rho_hidden=BEST_HYPERPARAMS['rho_hidden'],
        pool=BEST_HYPERPARAMS['pool'],
        dropout=BEST_HYPERPARAMS['dropout'],
        device=device,
        base_path=BASE_PATH,
        seed=SEED
    )

    # Training
    epoch_losses = model.train_from_data(
        detections=train_det,
        labels=train_lab,
        masks=train_mask,
        epochs=epochs,
        batch_size=batch_size,
        lr=lr,
        verbose=verbose
    )

    return model, epoch_losses

In [ ]:
def run_split_experiment(split_name: str, split_config: dict):
    """
    Run a complete training experiment for one split configuration.

    Args:
        split_name: Name of the experiment
        split_config: Dict with 'd3', 'd5', 'd7' proportions

    Returns:
        Dict with training metrics
    """
    print(f"\n{'='*70}")
    print(f"EXPERIMENT: {split_name}")
    print(f"{'='*70}")
    print(f"Split: d3={split_config['d3']*100:.0f}%, d5={split_config['d5']*100:.0f}%, d7={split_config['d7']*100:.0f}%")
    print(f"Hypothesis: {split_config['hypothesis']}")
    print(f"{'='*70}")

    start_time = time.time()

    # 1. Load combined dataset with masks
    print("\n[1/4] Loading dataset...")
    all_detections, all_labels, all_masks = load_mixed_flat_dataset_variable(split_config, TOTAL_SAMPLES)

    # 2. Split into train/val/test
    print("\n[2/4] Splitting dataset...")
    (train_det, train_lab, train_mask), (val_det, val_lab, val_mask), (test_det, test_lab, test_mask) = split_train_val_test(
        all_detections, all_labels, all_masks, seed=SEED
    )

    # 3. Train model
    print("\n[3/4] Training model...")
    model, epoch_losses = train_deepsets(
        train_det, train_lab, train_mask,
        epochs=EPOCHS,
        batch_size=BEST_HYPERPARAMS['batch_size'],
        lr=BEST_HYPERPARAMS['learning_rate'],
        verbose=True
    )

    # 4. Evaluate
    print("\n[4/4] Evaluating model...")
    train_acc = evaluate_deepsets(model, train_det[:10000], train_lab[:10000], train_mask[:10000])  # Subsample for speed
    val_acc = evaluate_deepsets(model, val_det, val_lab, val_mask)
    test_acc = evaluate_deepsets(model, test_det, test_lab, test_mask)

    training_time = time.time() - start_time

    # Compile metrics
    metrics = {
        'epoch_losses': epoch_losses,
        'final_loss': epoch_losses[-1] if epoch_losses else None,
        'train_accuracy': train_acc,
        'val_accuracy': val_acc,
        'test_accuracy': test_acc,
        'training_time_seconds': training_time,
    }

    # Save model and results
    save_model(model, split_name, metrics)
    save_training_results(split_name, metrics)

    # Print summary
    print(f"\n{'='*70}")
    print(f"EXPERIMENT COMPLETE: {split_name}")
    print(f"{'='*70}")
    print(f"  Training time: {training_time/60:.1f} minutes")
    print(f"  Final loss: {epoch_losses[-1]:.4f}" if epoch_losses else "  Final loss: N/A")
    print(f"  Train accuracy: {train_acc*100:.2f}%")
    print(f"  Val accuracy: {val_acc*100:.2f}%")
    print(f"  Test accuracy: {test_acc*100:.2f}%")
    print(f"{'='*70}")

    # Clean up memory
    del model, train_det, train_lab, train_mask, val_det, val_lab, val_mask, test_det, test_lab, test_mask, all_detections, all_labels, all_masks
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

    return metrics

## Run Training Experiments

In [ ]:
# Confirm hyperparameters
print(f"Using hyperparameters from tuning:")
for k, v in BEST_HYPERPARAMS.items():
    print(f"  {k}: {v}")

print(f"\n{'#'*70}")
print(f"STARTING DEEPSETS EXTRAPOLATION TRAINING EXPERIMENTS")
print(f"{'#'*70}")
print(f"Experiments to run: {len(experiments_to_run)}")
for exp in experiments_to_run:
    print(f"  - {exp}")

In [ ]:
# Run all selected experiments
all_results = {}

for i, split_name in enumerate(experiments_to_run):
    print(f"\n\n{'#'*70}")
    print(f"EXPERIMENT {i+1}/{len(experiments_to_run)}: {split_name}")
    print(f"{'#'*70}")

    split_config = SPLIT_EXPERIMENTS[split_name]
    metrics = run_split_experiment(split_name, split_config)
    all_results[split_name] = metrics

print(f"\n\n{'#'*70}")
print(f"ALL EXPERIMENTS COMPLETE")
print(f"{'#'*70}")

## Results Summary

In [ ]:
# Create summary table
if all_results:
    summary_data = []
    for split_name, metrics in all_results.items():
        config = SPLIT_EXPERIMENTS[split_name]
        summary_data.append({
            'Split': split_name,
            'd3 %': config['d3'] * 100,
            'd5 %': config['d5'] * 100,
            'd7 %': config['d7'] * 100,
            'Train Acc': metrics['train_accuracy'] * 100,
            'Val Acc': metrics['val_accuracy'] * 100,
            'Test Acc': metrics['test_accuracy'] * 100,
            'Final Loss': metrics['final_loss'],
            'Time (min)': metrics['training_time_seconds'] / 60,
        })

    df_summary = pd.DataFrame(summary_data)
    print("\nTraining Results Summary:")
    print(df_summary.to_string(index=False))

    # Save summary
    summary_path = RESULTS_DIR / "training_summary.csv"
    df_summary.to_csv(summary_path, index=False)
    print(f"\nSummary saved to: {summary_path}")

In [ ]:
# Plot training curves
if all_results:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Loss curves
    ax1 = axes[0]
    for split_name, metrics in all_results.items():
        if metrics['epoch_losses']:
            ax1.plot(metrics['epoch_losses'], label=split_name)
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.set_title('Training Loss by Split Configuration')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    # Accuracy comparison
    ax2 = axes[1]
    split_names = list(all_results.keys())
    val_accs = [all_results[s]['val_accuracy'] * 100 for s in split_names]
    test_accs = [all_results[s]['test_accuracy'] * 100 for s in split_names]

    x = np.arange(len(split_names))
    width = 0.35
    ax2.bar(x - width/2, val_accs, width, label='Validation')
    ax2.bar(x + width/2, test_accs, width, label='Test')
    ax2.set_ylabel('Accuracy (%)')
    ax2.set_title('Accuracy by Split Configuration')
    ax2.set_xticks(x)
    ax2.set_xticklabels(split_names, rotation=45, ha='right')
    ax2.legend()
    ax2.grid(True, alpha=0.3, axis='y')

    plt.tight_layout()

    # Save plot
    plot_path = PLOTS_DIR / "training_results.png"
    plt.savefig(plot_path, dpi=150, bbox_inches='tight')
    print(f"Plot saved to: {plot_path}")

    plt.show()

## Next Steps

After training completes:
1. Review the training results summary above
2. Run `testing.ipynb` to evaluate extrapolation to d=9, d=11, and d=13
3. Compare how different split configurations affect extrapolation performance
4. Compare DeepSets results against GraphSAGE results for the same splits